# Analyse: visuelle Filter


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm
from transformers import AutoImageProcessor, AutoModelForImageClassification

DATA_DIR = Path('../../data')
COMBINED_CSV = DATA_DIR / '03_datasets/influencer_balanced/01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50.csv'
OUTPUT_CSV = DATA_DIR / '04_analysis_results' / 'visual_features' / '15_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_visual_filter.csv'
FRAME_ROOT = DATA_DIR / '02_media/stratified_sample/frames'

SOURCE_FILTER = None
DEFAULT_MAX_FRAMES_PER_VIDEO = 12
MODEL_NAME = 'romitbarua/autotrain-deepfakeface_only_faces_insightface-94902146221'

device = 'mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

df = pd.read_csv(COMBINED_CSV)
if SOURCE_FILTER:
    df = df[df['influencer_type'].isin(SOURCE_FILTER)].copy()

video_id_col = 'video_id' if 'video_id' in df.columns else 'id'
df[video_id_col] = df[video_id_col].astype(str)
print(f'Loaded {len(df)} rows from {COMBINED_CSV.name}')


c:\Users\hanna\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
Loaded 500 rows from 01_AI_AND_REAL_TIKTOK_VIDEOS_stratified_per_influencer_50.csv


In [2]:
METRIC_COLUMNS = [
    'filter_strength_prob',
    'filter_strength_label',
    'filter_strength_std',
    'processed_frame_count',
]

def has_frames(video_id: str) -> bool:
    return (FRAME_ROOT / video_id).is_dir()

df['has_frames'] = [has_frames(video_id) for video_id in df[video_id_col].astype(str)]
missing_ids = df.loc[~df['has_frames'], video_id_col].astype(str).tolist()
print(f"Videos with frame folder: {df['has_frames'].sum()}")
print(f'Videos missing frame folder: {len(missing_ids)}')
if missing_ids:
    print('First missing video_ids:', missing_ids[:20])

df = df[df['has_frames']].reset_index(drop=True)
print(f'Restricted to {len(df)} rows with available frames')

processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForImageClassification.from_pretrained(MODEL_NAME).to(device).eval()
id2label = getattr(model.config, 'id2label', None)
print('Model classes:', id2label)


Videos with frame folder: 500
Videos missing frame folder: 0
Restricted to 500 rows with available frames


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
c:\Users\hanna\AppData\Local\Programs\Python\Python311\Lib\site-packages\transformers\models\vit\feature_extraction_vit.py:30: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(


Model classes: {0: 'insight', 1: 'real'}


In [3]:
def get_duration_seconds(row: pd.Series):
    for col in ('video_duration_seconds', 'duration_seconds', 'duration', 'video_duration'):
        if col in row.index:
            value = row.get(col, np.nan)
            if pd.notna(value):
                return value
    return np.nan

def sample_frame_paths(video_id: str, duration_seconds=None, default_max_frames: int = DEFAULT_MAX_FRAMES_PER_VIDEO):
    folder = FRAME_ROOT / video_id
    if not folder.is_dir():
        return []

    frame_files = sorted(folder.glob('*.jpeg'))
    if not frame_files:
        frame_files = sorted(folder.glob('*.jpg'))
    if not frame_files:
        return []

    try:
        duration_value = float(duration_seconds)
    except (TypeError, ValueError):
        duration_value = np.nan

    if pd.notna(duration_value) and duration_value > 0:
        target_frames = int(np.ceil(duration_value))
    else:
        target_frames = default_max_frames

    target_frames = max(1, min(target_frames, default_max_frames, len(frame_files)))
    indices = np.linspace(0, len(frame_files) - 1, num=target_frames, dtype=int)
    return [frame_files[idx] for idx in np.unique(indices)]

def detect_filter_strength(image_path: Path) -> float:
    try:
        image = Image.open(image_path).convert('RGB')
        inputs = processor(images=image, return_tensors='pt').to(device)
        with torch.no_grad():
            logits = model(**inputs).logits
            probs = F.softmax(logits, dim=1).cpu().numpy()[0]

        # Bestehende Modellkonvention in diesem Ordner:
        # Index 1 wird als gefiltert, synthetisch oder artefaktstark interpretiert.
        return float(probs[1]) if len(probs) > 1 else float(probs[0])
    except Exception as exc:
        print(f'Error reading {image_path}: {exc}')
        return np.nan

def categorize_prob(p: float) -> str:
    if np.isnan(p):
        return 'Unbestimmt'
    if p < 0.30:
        return 'keine'
    if p < 0.65:
        return 'leicht'
    return 'stark'


In [4]:
def extract_visual_filter_metrics(row: pd.Series) -> dict:
    video_id = row[video_id_col]
    duration_seconds = get_duration_seconds(row)
    frame_paths = sample_frame_paths(video_id, duration_seconds=duration_seconds)
    probs = [detect_filter_strength(frame_path) for frame_path in frame_paths]
    probs = [p for p in probs if pd.notna(p)]

    metrics = {'video_id': video_id}
    for col in METRIC_COLUMNS:
        metrics[col] = np.nan

    metrics['processed_frame_count'] = len(probs)

    if probs:
        values = np.asarray(probs, dtype=float)
        mean_prob = float(values.mean())
        metrics['filter_strength_prob'] = mean_prob
        metrics['filter_strength_std'] = float(values.std())
        metrics['filter_strength_label'] = categorize_prob(mean_prob)
    else:
        metrics['filter_strength_label'] = 'Unbestimmt'

    return metrics

result_rows = [extract_visual_filter_metrics(row) for _, row in tqdm(df.iterrows(), total=len(df))]
feature_df = pd.DataFrame(result_rows)
merged = df.merge(feature_df, on='video_id', how='left') if 'video_id' in df.columns else df.merge(feature_df, left_on=video_id_col, right_on='video_id', how='left')

OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
merged.to_csv(OUTPUT_CSV, index=False)

print(merged[['influencer_type', 'filter_strength_prob', 'filter_strength_std', 'processed_frame_count']].groupby('influencer_type').agg(['count', 'mean', 'std']).round(4))
print(merged.groupby(['influencer_type', 'filter_strength_label']).size().unstack(fill_value=0))
print(f'Wrote {OUTPUT_CSV} with shape {merged.shape}')
merged[[video_id_col, 'influencer_type', 'filter_strength_label', 'filter_strength_prob', 'processed_frame_count']].head()


100%|██████████| 500/500 [08:23<00:00,  1.01s/it]

                filter_strength_prob                 filter_strength_std  \
                               count    mean     std               count   
influencer_type                                                            
ai                               250  0.9924  0.0160                 250   
real                             250  0.9889  0.0299                 250   

                                processed_frame_count                  
                   mean     std                 count    mean     std  
influencer_type                                                        
ai               0.0101  0.0369                   250  10.268  2.1180  
real             0.0151  0.0449                   250  11.248  1.5762  
filter_strength_label  stark
influencer_type             
ai                       250
real                     250
Wrote ..\..\data\04_analysis_results\visual_features\15_AI_AND_REAL_TIKTOK_VIDEOS_stratified_with_visual_filter.csv with shape (500, 49)


,video_id,influencer_type,filter_strength_label,filter_strength_prob,processed_frame_count
0,7516243181650988334,ai,stark,0.996765,7
1,7515925552549678378,ai,stark,0.991559,10
2,7521159314757684535,ai,stark,0.988896,8
3,7521486299098778894,ai,stark,0.996831,8
4,7521490969468865847,ai,stark,0.991135,8
